In [ ]:
"""
06_residual_analysis.py -- ScoutAI Statistical Diagnostics & Residuals

Evaluates model health by analyzing the residual errors (Actual Value - Predicted Value)
for both the Full and Performance-Only models. Automatically generates
heteroscedasticity and normality plots, saving them to the images directory.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
import matplotlib
matplotlib.use('Agg')  # Headless-safe
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

# Load environment variables
load_dotenv(find_dotenv())

MODELS_DIR = "models"
IMAGES_DIR = "images"
DATA_DIR = "data"

# Ensure output directories exist
for directory in [MODELS_DIR, IMAGES_DIR, DATA_DIR]:
    os.makedirs(directory, exist_ok=True)

def main():
    # ==========================================
    # 1. DATABASE CONNECTION & DATA LOAD
    # ==========================================
    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Fetching data for statistical diagnostics...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        df = pd.read_sql("SELECT * FROM view_scout_master", engine)
    except Exception as e:
        print(f"[ERROR] Database connection failed: {e}")
        sys.exit(1)

    # Clean the target variable (keep only players with a valid market value)
    df['current_market_value'] = pd.to_numeric(df['current_market_value'], errors='coerce').fillna(0)
    df = df[df['current_market_value'] > 0].copy()

    # ==========================================
    # 2. FEATURE ENGINEERING
    # ==========================================
    print("[SYSTEM] Engineering features...")
    cols_to_clean = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'max_career_transfer_fee', 'most_recent_transfer_fee', 'height_in_cm',
        'minutes_per_match', 'total_goals', 'total_assists', 'goals_per_90',
        'assists_per_90', 'total_yellow_cards', 'total_red_cards', 'club_squad_size',
        'club_avg_age', 'stadium_seats', 'foreigners_percentage', 'contract_months_remaining',
    ]
    for col in cols_to_clean:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')

    league_weights = {
        'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
        'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
    }
    df['league_coefficient'] = df['competition_name'].map(league_weights).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    FULL_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
        'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
        'detailed_position', 'foot', 'passport_tier',
    ]

    PERFORMANCE_FEATURES = [
        'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
        'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
        'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
        'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
        'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
    ]

    # ==========================================
    # 3. PREDICTIONS & RESIDUAL CALCULATION
    # ==========================================
    print("[SYSTEM] Loading models and calculating residuals...")
    try:
        model_full = joblib.load(os.path.join(MODELS_DIR, 'scout_model_full.pkl'))
        model_perf = joblib.load(os.path.join(MODELS_DIR, 'scout_model_performance_only.pkl'))
    except FileNotFoundError as e:
        print(f"[ERROR] Could not load models from '{MODELS_DIR}'. Please train models first.")
        sys.exit(1)

    df['predicted_market_value'] = np.nan
    df['model_used'] = ""

    has_history_mask = df['has_transfer_history'] == 1
    no_history_mask = ~has_history_mask

    if has_history_mask.any():
        log_preds_full = model_full.predict(df.loc[has_history_mask, FULL_FEATURES])
        df.loc[has_history_mask, 'predicted_market_value'] = np.expm1(log_preds_full)
        df.loc[has_history_mask, 'model_used'] = 'full'

    if no_history_mask.any():
        log_preds_perf = model_perf.predict(df.loc[no_history_mask, PERFORMANCE_FEATURES])
        df.loc[no_history_mask, 'predicted_market_value'] = np.expm1(log_preds_perf)
        df.loc[no_history_mask, 'model_used'] = 'performance_only'

    # Residual = Actual Value - Predicted Value
    df['residuals'] = df['current_market_value'] - df['predicted_market_value']

    # ==========================================
    # 4. VISUALIZATION
    # ==========================================
    print("[SYSTEM] Generating diagnostic plots...")
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    model_colors = {'full': '#e74c3c', 'performance_only': '#9b59b6'}
    hist_colors = {'full': '#3498db', 'performance_only': '#2ecc71'}

    for row, label in enumerate(['full', 'performance_only']):
        subset = df[df['model_used'] == label]

        # Residuals vs Predicted Values
        sns.scatterplot(
            x=subset['predicted_market_value'] / 1e6,
            y=subset['residuals'] / 1e6,
            alpha=0.5,
            color=model_colors[label],
            edgecolor='k',
            ax=axes[row, 0],
        )
        axes[row, 0].axhline(y=0, color='black', linestyle='--', linewidth=2)
        axes[row, 0].set_title(f"Residuals vs. Predicted -- {label.replace('_', ' ').title()} model (n={len(subset)})", fontsize=13, fontweight='bold')
        axes[row, 0].set_xlabel("Predicted Market Value (Million €)", fontsize=11)
        axes[row, 0].set_ylabel("Residual Error (Million €)", fontsize=11)

        # Distribution of residuals
        sns.histplot(
            subset['residuals'] / 1e6,
            kde=True,
            bins=50,
            color=hist_colors[label],
            ax=axes[row, 1],
        )
        axes[row, 1].axvline(x=0, color='black', linestyle='--', linewidth=2)
        axes[row, 1].set_title(f"Error Distribution -- {label.replace('_', ' ').title()} model", fontsize=13, fontweight='bold')
        axes[row, 1].set_xlabel("Residual Error (Million €)", fontsize=11)
        axes[row, 1].set_ylabel("Frequency (Number of Players)", fontsize=11)

    plt.suptitle("ScoutAI Diagnostics: Model Health & Variance Analysis (Full vs. Performance-Only)",
                 fontsize=17, fontweight='bold', y=1.01)
    plt.tight_layout()
    
    plot_path = os.path.join(IMAGES_DIR, 'residual_analysis.png')
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[SUCCESS] Diagnostic plots saved to '{plot_path}'")

    # ==========================================
    # 5. QUICK NUMERIC SUMMARY & EXPORT
    # ==========================================
    report_lines = []
    report_lines.append("ScoutAI: Residual Analysis Summary\n")
    report_lines.append("==========================================\n")

    for label in ['full', 'performance_only']:
        subset = df[df['model_used'] == label]
        
        summary_text = (
            f"\n--- {label.replace('_', ' ').title()} Model Residual Summary ---\n"
            f"  n               : {len(subset)}\n"
            f"  mean residual   : €{subset['residuals'].mean():,.0f}\n"
            f"  median residual : €{subset['residuals'].median():,.0f}\n"
            f"  std residual    : €{subset['residuals'].std():,.0f}\n"
        )
        
        print(summary_text, end="")
        report_lines.append(summary_text)

    # Save numeric summary to TXT in DATA_DIR
    txt_export_path = os.path.join(DATA_DIR, 'residual_summary.txt')
    with open(txt_export_path, 'w', encoding='utf-8') as f:
        f.writelines(report_lines)
    
    print(f"\n[SUCCESS] Numeric summary saved to '{txt_export_path}'")

if __name__ == "__main__":
    main()

[SYSTEM] Fetching data for statistical diagnostics...
[SYSTEM] Engineering features...
[SYSTEM] Loading models and calculating residuals...
[SYSTEM] Generating diagnostic plots...
[SUCCESS] Diagnostic plots saved to 'images\residual_analysis.png'

--- Full Model Residual Summary ---
  n               : 9334
  mean residual   : €402,938
  median residual : €2,319
  std residual    : €3,024,876

--- Performance Only Model Residual Summary ---
  n               : 11046
  mean residual   : €156,616
  median residual : €-12,578
  std residual    : €1,210,137

[SUCCESS] Numeric summary saved to 'data\residual_summary.txt'
